In [3]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Your existing settings
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "practice_db"
DB_USER = "postgres"
DB_PASS = "mysecretpassword"

conn_str = f"postgresql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)

def run_sql(query: str):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(query))
            df = pd.DataFrame(result.mappings())   # ← this handles SQLAlchemy 2.0+
            display(df)
    except Exception as e:
        print(f"Error executing query:\n{e}")

print("✅ Fixed SQL execution environment ready!")

✅ Fixed SQL execution environment ready!


In [ ]:
sql = """
SELECT e.name, d.dept_name, e.salary
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
ORDER BY salary DESC
LIMIT 5;
"""

run_sql(sql)


In [ ]:
sql = """ select * from employees limit 5"""
run_sql(sql)

In [ ]:
sql="""
-- Exercise 1 ── All three ranking functions side-by-side
-- Show name, dept, salary, row_number, rank, dense_rank per department
-- Expected: Alice & Bob both dense_rank=1, rank=1, but row_number different
select e.name, d.dept_name, e.salary 
,ROW_NUMBER() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS row_n
,RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS rnk
,DENSE_RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS dns_rnk
from employees e JOIN departments d ON e.dept_id = d.dept_id;
"""
run_sql(sql)

In [ ]:
sql= """
-- Exercise 2 ── Top-2 earners per department (use DENSE_RANK + CTE)
-- Return name, dept_name, salary, rank — only rank <= 2
WITH ranked AS (select e.name as name, d.dept_name as dept_name, e.salary as salary
,DENSE_RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS rank
from employees e JOIN departments d ON e.dept_id = d.dept_id) 
SELECT * FROM ranked WHERE rank <= 2;
"""

run_sql(sql)


In [ ]:
sql="""
-- Exercise 3 ── Day-over-day CPU delta + percentage change
-- Columns: server_id, date, cpu, prev_cpu, delta, pct_change
-- Hint: LAG + (current - prev)/prev * 100
--select * from telemetry limit 5;
WITH cpu_calc AS(
select server_id, collection_date as date, cpu_utilization as cpu 
,LAG(cpu_utilization, 1) OVER ( PARTITION BY server_id ORDER BY collection_date ) AS prev_cpu
from telemetry) 
select 
server_id , date, cpu , prev_cpu,
(cpu - prev_cpu) as delta,
100 * (cpu - prev_cpu) / prev_cpu  as pct_change
from cpu_calc ;
"""

run_sql(sql)

In [ ]:
sql= """
-- Exercise 4 ── Servers with continuous 3-day upward CPU trend
-- (today > yesterday AND yesterday > day-before-yesterday)
-- Return server_id, date, cpu, prev1, prev2
WITH cpu_calc AS(
select server_id, collection_date as date, cpu_utilization as cpu 
,LAG(cpu_utilization, 1) OVER ( PARTITION BY server_id ORDER BY collection_date ) AS yesterday_cpu
,LAG(cpu_utilization, 2) OVER ( PARTITION BY server_id ORDER BY collection_date ) AS day_before_yesterday_cpu
from telemetry) 
select server_id , date, cpu, yesterday_cpu as prev1, day_before_yesterday_cpu as prev2 from cpu_calc
where 
cpu > yesterday_cpu AND yesterday_cpu > day_before_yesterday_cpu;
"""
run_sql(sql)


In [ ]:
sql="""
-- Exercise 5 ── 7-day rolling average & running total CPU per server
-- Use ROWS BETWEEN 6 PRECEDING AND CURRENT ROW

select server_id, collection_date as date, cpu_utilization as cpu,
AVG(cpu_utilization) OVER 
( PARTITION BY server_id ORDER BY collection_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW ) AS avg_7day, 
SUM(cpu_utilization) OVER 
( PARTITION BY server_id ORDER BY collection_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW ) AS sum_7day

from 
telemetry
"""
run_sql(sql)

In [ ]:
sql="""
-- Exercise 6 ── Employees earning above their department average
-- No subquery — use window AVG() + CTE or inline
WITH avg_calc AS(
select e.name, d.dept_name, e.salary 
,AVG(e.salary) OVER (PARTITION BY e.dept_id ) AS dept_avg
from employees e JOIN departments d ON e.dept_id = d.dept_id)
select name, dept_name ,
salary, ROUND(dept_avg, 2) as dept_avg
from avg_calc
WHERE
salary >= dept_avg
ORDER BY dept_name, salary DESC;
"""
run_sql(sql)

In [ ]:
sql="""
-- Exercise 7 ── Salary percentile rank within department (PERCENT_RANK)
-- 0 = lowest, 1 = highest in dept

select e.name, d.dept_name, e.salary 
,ROUND( PERCENT_RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary ASC)::numeric , 3)
from employees e JOIN departments d ON e.dept_id = d.dept_id

"""

run_sql(sql)

In [ ]:
sql="""
-- Exercise 7 ── Salary percentile rank within department (PERCENT_RANK)
-- 0 = lowest, 1 = highest in dept

select e.name, d.dept_name, e.salary 
,100 * ROUND( PERCENT_RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary ASC)::numeric , 4) 
AS salary_percentile_0_to_100
from employees e JOIN departments d ON e.dept_id = d.dept_id

"""

run_sql(sql)

In [ ]:
sql="""
-- Exercise 8 ── Combine: rank + previous salary in same dept
-- Show name, salary, dept_rank (dense), prev_salary_in_dept (ordered by hire_date)
select * from employees limit 3;
"""
run_sql(sql)

In [ ]:
sql="""
-- Exercise 8 ── Combine: rank + previous salary in same dept
-- Show name, salary, dept_rank (dense), prev_salary_in_dept (ordered by hire_date)
--- select * from employees limit 3;
WITH ranked_and_lag AS(
SELECT 
        e.name,
        d.dept_name,
        e.salary,
        e.hire_date,
        DENSE_RANK() OVER (
            PARTITION BY e.dept_id 
            ORDER BY e.salary DESC
        ) AS dept_rank
        , LAG(e.salary) OVER (
        PARTITION BY e.dept_id 
        ORDER BY e.hire_date ASC 
        ) AS prev_salary_in_dept
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
)
SELECT 
    name,
    dept_name,
    salary,
    dept_rank,
    prev_salary_in_dept
FROM ranked_and_lag
ORDER BY dept_name, hire_date ASC;   -- show in hiring order for clarity
    
"""
run_sql(sql)

In [ ]:
sql ="""
SELECT 
        e.name,
        d.dept_name,
        e.salary,
--        e.hire_date,
        DENSE_RANK() OVER (
            PARTITION BY e.dept_id 
            ORDER BY e.salary DESC
        ) AS dept_rank
        , LAG(e.salary) OVER (
        PARTITION BY e.dept_id 
        ORDER BY e.hire_date ASC 
        ) AS prev_salary_in_dept
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
ORDER BY dept_name, hire_date ASC;
"""
run_sql(sql)

In [ ]:
sql="""
-- Exercise 9 ── Running total CPU hours per server (cumulative sum)
-- Reset per server, ordered by date
SELECT
    server_id, collection_date, cpu_utilization,
    ROUND(
    SUM(cpu_utilization) OVER (
    PARTITION BY server_id 
    ORDER BY collection_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 1)
    AS running_total
FROM telemetry
ORDER BY server_id, collection_date;




"""
run_sql(sql)

In [ ]:
sql="""
-- Exercise 9 ── Running total CPU hours per server (cumulative sum)
-- Reset per server, ordered by date
SELECT
    server_id, collection_date, cpu_utilization,
    ROUND(
    SUM(cpu_utilization) OVER (
    PARTITION BY server_id 
    ORDER BY collection_date

    ), 1)
    AS running_total
FROM telemetry
ORDER BY server_id, collection_date;




"""
run_sql(sql)

In [11]:
sql = """
-- Exercise 10 (challenge) ── Detect first time CPU crossed 80% per server
-- Use window + conditional logic or LEAD
With ranked_lag AS (
WITH cpu_lag as (
SELECT  server_id, collection_date, cpu_utilization
, LAG (cpu_utilization,1 ) OVER (PARTITION BY server_id 
                                ORDER BY collection_date ) CPU_yesterday
FROM telemetry)
Select server_id , Collection_date, cpu_utilization , cpu_yesterday,
RANK()  OVER (PARTITION BY server_id  ORDER BY collection_date )  as rnk
From 
cpu_lag
Where 
cpu_utilization >= 80 AND
(cpu_yesterday < 80 OR cpu_yesterday IS NULL)
Order by collection_date ASC)
SELECT server_id, collection_date, rnk, cpu_utilization, cpu_yesterday
FROM ranked_lag
WHERE rnk = 1
Order by server_id, rnk


"""
run_sql(sql)

,collection_date,cpu_utilization,cpu_yesterday,rnk,server_id
0,2026-01-03,91.4,79.6,1,srv-01
1,2026-01-02,90.0,47.2,1,srv-02
2,2026-01-02,87.3,44.6,1,srv-03
3,2026-01-02,87.3,43.6,1,srv-04
4,2026-01-03,93.7,79.1,1,srv-05
5,2026-01-02,83.5,49.0,1,srv-06


In [14]:
sql = """
-- Exercise 10 (challenge) ── Detect first time CPU crossed 80% per server
-- Use window + conditional logic or LEAD
WITH cpu_with_lag AS (
    SELECT
        server_id,
        collection_date,
        cpu_utilization,
        LAG(cpu_utilization) OVER (
            PARTITION BY server_id 
            ORDER BY collection_date
        ) AS prev_cpu
    FROM telemetry
),
crossings AS (
    SELECT
        server_id,
        collection_date,
        cpu_utilization,
        prev_cpu,
        ROW_NUMBER() OVER (
            PARTITION BY server_id 
            ORDER BY collection_date
        ) AS rn
    FROM cpu_with_lag
    WHERE cpu_utilization >= 80
      AND (prev_cpu < 80 OR prev_cpu IS NULL)
)
SELECT
    server_id,
    collection_date AS first_cross_date,
    cpu_utilization AS cpu_at_cross,
    prev_cpu AS prev_day_cpu
FROM crossings
WHERE rn = 1
ORDER BY server_id;
"""

run_sql(sql)

,cpu_at_cross,first_cross_date,prev_day_cpu,server_id
0,91.4,2026-01-03,79.6,srv-01
1,90.0,2026-01-02,47.2,srv-02
2,87.3,2026-01-02,44.6,srv-03
3,87.3,2026-01-02,43.6,srv-04
4,93.7,2026-01-03,79.1,srv-05
5,83.5,2026-01-02,49.0,srv-06


In [13]:
sql = """
-- Exercise 10 (challenge) ── Detect first time CPU crossed 80% per server
-- Use window + conditional logic or LEAD

SELECT 
    server_id,
    MIN(CASE WHEN cpu_utilization >= 80 THEN collection_date END) AS first_cross_date,
    MIN(CASE WHEN cpu_utilization >= 80 THEN cpu_utilization END) AS cpu_at_cross
FROM telemetry
GROUP BY server_id
HAVING MIN(CASE WHEN cpu_utilization >= 80 THEN collection_date END) IS NOT NULL;

"""

run_sql(sql)

,cpu_at_cross,first_cross_date,server_id
0,83.5,2026-01-02,srv-06
1,81.0,2026-01-02,srv-04
2,81.9,2026-01-02,srv-03
3,83.3,2026-01-03,srv-05
4,82.2,2026-01-02,srv-02
5,86.1,2026-01-03,srv-01
